# Example 05A: Prepare Ensemble Siblings

This notebook builds the physical SewerTris sibling projects only. It runs each realization through step 9, then writes a manifest that the simulation notebook can resume from.

Recommended split:
- Notebook 05A: steps 1-9, project/model generation. Fast enough to run many siblings in parallel.
- Notebook 05B: steps 10-12, SWMM inputs, simulation, and flow-component extraction. This is the expensive stage and should be restarted independently.

## Setup

In [7]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import json
import os
import subprocess
import sys
import time

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "sewertris").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

EXAMPLES_DIR = PROJECT_ROOT / "examples"
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import sewertris as sp

BASE_PROJECT_FILE = EXAMPLES_DIR / "output_example_2_project" / "sewertris_project.json"
ENSEMBLE_ROOT = EXAMPLES_DIR / "output_example_5_ensembles"
SCENARIO_NAME = "bwf_gwi_rdii"

base_project = sp.SewerTrisProject.load(BASE_PROJECT_FILE)
ENSEMBLE_ROOT.mkdir(parents=True, exist_ok=True)
print("Base project:", base_project.project_file)

Base project: output_example_2_project/sewertris_project.json


## Ensemble Definition

Keep the ensemble definition here so the physical projects and later simulations use the same realization names and output folders.

In [8]:
N_REALIZATIONS = 3
BASE_LAYOUT_SEED = 1000
BASE_RAINFALL_SEED = 42

RUN_PREPARATION = True
USE_PARALLEL = True
MAX_WORKERS = min(10, N_REALIZATIONS * 2)
PREPARED_MANIFEST = ENSEMBLE_ROOT / "prepared_siblings_manifest.csv"

DRAINAGE_SHIFT_TOPOGRAPHY = {
    "min_elevation": 280,
    "max_elevation": 290,
    "cell_size": 10,
    "outlet_direction": "W",
    "smoothing_factor": 1,
}


# def seed_changes(i):
#     return {
#         "layout_seed": BASE_LAYOUT_SEED + i,
#         "rainfall": {
#             "regenerate": True,
#             "random_seed": BASE_RAINFALL_SEED + i,
#         },
#     }


def seed_changes(i):
    return {
        "layout_seed": BASE_LAYOUT_SEED + i,
        "rainfall": {
            "random_seed": BASE_RAINFALL_SEED + i,
        },
    }

sibling_specs = []
for i in range(1, N_REALIZATIONS + 1):
    sibling_specs.append(
        {
            "ensemble": "seed_only",
            "realization": i,
            "name": f"Stillwater seed-only realization {i:02d}",
            "output_dir": ENSEMBLE_ROOT / "seed_only" / f"run_{i:03d}",
            "changes": seed_changes(i),
        }
    )

for i in range(1, N_REALIZATIONS + 1):
    changes = seed_changes(i)
    changes["topography_config"] = DRAINAGE_SHIFT_TOPOGRAPHY
    sibling_specs.append(
        {
            "ensemble": "drainage_shift_west",
            "realization": i,
            "name": f"Stillwater west-drainage realization {i:02d}",
            "output_dir": ENSEMBLE_ROOT / "drainage_shift_west" / f"run_{i:03d}",
            "changes": changes,
        }
    )

for spec in sibling_specs:
    stem = f"prepare_{spec['ensemble']}_{spec['realization']:03d}"
    spec["progress_path"] = ENSEMBLE_ROOT / "_workers" / f"{stem}_progress.json"

pd.DataFrame(
    {
        "ensemble": [spec["ensemble"] for spec in sibling_specs],
        "realization": [spec["realization"] for spec in sibling_specs],
        "output_dir": [str(spec["output_dir"]) for spec in sibling_specs],
        "rerun_keys": [sorted(spec["changes"]) for spec in sibling_specs],
    }
)

,ensemble,realization,output_dir,rerun_keys
0,seed_only,1,/Users/gjperezm/Library/CloudStorage/OneDrive-...,"[layout_seed, rainfall]"
1,seed_only,2,/Users/gjperezm/Library/CloudStorage/OneDrive-...,"[layout_seed, rainfall]"
2,seed_only,3,/Users/gjperezm/Library/CloudStorage/OneDrive-...,"[layout_seed, rainfall]"
3,drainage_shift_west,1,/Users/gjperezm/Library/CloudStorage/OneDrive-...,"[layout_seed, rainfall, topography_config]"
4,drainage_shift_west,2,/Users/gjperezm/Library/CloudStorage/OneDrive-...,"[layout_seed, rainfall, topography_config]"
5,drainage_shift_west,3,/Users/gjperezm/Library/CloudStorage/OneDrive-...,"[layout_seed, rainfall, topography_config]"


## Parallel Preparation

Each worker clones the base project and reruns only through step 9. Step 10 inputs, SWMM execution, and output decomposition are left untouched for notebook 05B.

In [9]:
def json_ready(value):
    if isinstance(value, Path):
        return str(value.resolve())
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_ready(v) for v in value]
    return value


def run_worker_payload(payload, stem):
    worker_dir = ENSEMBLE_ROOT / "_workers"
    worker_dir.mkdir(parents=True, exist_ok=True)
    payload_path = worker_dir / f"{stem}.json"
    result_path = worker_dir / f"{stem}_result.json"
    payload_path.write_text(json.dumps(json_ready(payload), indent=2))

    env = os.environ.copy()
    env["PYTHONPATH"] = os.pathsep.join([str(SRC_DIR), env.get("PYTHONPATH", "")]).strip(os.pathsep)
    completed = subprocess.run(
        [
            sys.executable,
            "-m",
            "sewertris.ensemble",
            str(payload_path),
            "--result-file",
            str(result_path),
        ],
        cwd=PROJECT_ROOT,
        env=env,
        text=True,
        capture_output=True,
    )
    if completed.returncode != 0:
        raise RuntimeError(f"Worker failed for {stem}\nSTDOUT:\n{completed.stdout}\nSTDERR:\n{completed.stderr}")
    return json.loads(result_path.read_text())


def read_progress(path):
    path = Path(path)
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text())
    except json.JSONDecodeError:
        return None


def progress_label(spec):
    return f"{spec['ensemble']} run_{int(spec['realization']):03d}"


def print_active_progress(pending_specs, completed, successful, failed):
    total = completed + len(pending_specs)
    active = []
    for spec in pending_specs:
        event = read_progress(spec["progress_path"])
        if event is None:
            active.append(f"{progress_label(spec)}: queued")
        else:
            step = event.get("step", "starting")
            active.append(f"{progress_label(spec)}: {step}")
    preview = "; ".join(active[:6])
    if len(active) > 6:
        preview += f"; ... +{len(active) - 6} more"
    print(
        f"[progress] completed {completed}/{total} "
        f"(success={successful}, failed={failed}). Active: {preview}"
    )


def run_parallel(items, worker, max_workers, heartbeat_seconds=10):
    results = []
    successful = 0
    failed = 0
    last_heartbeat = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(worker, item): item for item in items}
        pending = set(futures)

        while pending:
            finished = [future for future in pending if future.done()]
            if finished:
                for future in finished:
                    pending.remove(future)
                    spec = futures[future]
                    try:
                        result = future.result()
                    except Exception:
                        failed += 1
                        print(f"[failed] {progress_label(spec)}")
                        raise
                    else:
                        successful += 1
                        results.append(result)
                        event = read_progress(spec["progress_path"]) or {}
                        print(
                            f"[done] {progress_label(spec)} "
                            f"({successful}/{len(items)} successful) "
                            f"last_step={event.get('step', result.get('stop_after_step'))}"
                        )
                continue

            now = time.time()
            if now - last_heartbeat >= heartbeat_seconds:
                pending_specs = [futures[future] for future in pending]
                print_active_progress(
                    pending_specs,
                    completed=len(items) - len(pending),
                    successful=successful,
                    failed=failed,
                )
                last_heartbeat = now
            time.sleep(1)

    return results


def prepare_sibling(spec):
    stem = f"prepare_{spec['ensemble']}_{spec['realization']:03d}"
    payload = {
        "parent_project_file": BASE_PROJECT_FILE,
        "scenario_name": SCENARIO_NAME,
        "run_flow_components": False,
        "stop_after_step": 9,
        "progress_path": spec["progress_path"],
        "spec": {key: value for key, value in spec.items() if key != "progress_path"},
    }
    return run_worker_payload(payload, stem)


if RUN_PREPARATION:
    if USE_PARALLEL:
        preparation_results = run_parallel(sibling_specs, prepare_sibling, MAX_WORKERS)
    else:
        preparation_results = [prepare_sibling(spec) for spec in sibling_specs]

    prepared_manifest = pd.DataFrame(preparation_results).sort_values(["ensemble", "realization"])
    prepared_manifest.to_csv(PREPARED_MANIFEST, index=False)
    display(prepared_manifest)
else:
    print("Set RUN_PREPARATION = True to prepare the ensemble siblings.")

[progress] completed 0/6 (success=0, failed=0). Active: seed_only run_001: 03_stochastic_tetris_completion; seed_only run_003: 03_stochastic_tetris_completion; drainage_shift_west run_003: 03_stochastic_tetris_completion; drainage_shift_west run_001: 03_stochastic_tetris_completion; drainage_shift_west run_002: 03_stochastic_tetris_completion; seed_only run_002: 03_stochastic_tetris_completion
[failed] seed_only run_001


RuntimeError: Worker failed for prepare_seed_only_001
STDOUT:

STDERR:
/opt/anaconda3/envs/sewertris/lib/python3.10/runpy.py:126: RuntimeWarning: 'sewertris.ensemble' found in sys.modules after import of package 'sewertris', but prior to execution of 'sewertris.ensemble'; this may result in unpredictable behaviour
  warn(RuntimeWarning(msg))
Traceback (most recent call last):
  File "/opt/anaconda3/envs/sewertris/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/anaconda3/envs/sewertris/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/Users/gjperezm/Library/CloudStorage/OneDrive-OklahomaAandMSystem/Academia/Codes/SewerTris_V1/sewertris/src/sewertris/ensemble.py", line 157, in <module>
    raise SystemExit(main())
  File "/Users/gjperezm/Library/CloudStorage/OneDrive-OklahomaAandMSystem/Academia/Codes/SewerTris_V1/sewertris/src/sewertris/ensemble.py", line 147, in main
    result = r

## Prepared Projects

In [ ]:
if PREPARED_MANIFEST.exists():
    prepared_manifest = pd.read_csv(PREPARED_MANIFEST)
    display(prepared_manifest)
else:
    print("No prepared-sibling manifest yet.")